In [1]:
import pandas as pd
import numpy as np
import sklearn
import dvc
import mlflow

print("Pandas : ", pd.__version__)
print("Numpy : ", np.__version__)
print("Scikit-Learn : ", sklearn.__version__)
print("MLFlow : ", mlflow.__version__)

C:\Users\opc\custchurn-mlops\mlops_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pandas :  3.1.1
Numpy :  2.4.6
Scikit-Learn :  1.9.1
MLFlow :  3.18.0


In [2]:
import os
print (os.getcwd())

C:\Users\opc\custchurn-mlops


In [3]:
from sklearn.model_selection import train_test_split  # 7043 rows - training set, testing set
from sklearn.preprocessing import StandardScaler, OneHotEncoder # dataset - cols - higher values (area)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report

In [4]:
df = pd.read_csv(r"C:\Users\opc\data\WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Dataset loaded successfully")
print("Dataset shape : ", df.shape)
df.head() # prints the first 5 rows of the dataset

Dataset loaded successfully
Dataset shape :  (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
# Churn Percentage  : churn col - yes no
print(df["Churn"].value_counts())
print("Churn Percentage")
print(df["Churn"].value_counts(normalize=True) * 100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn Percentage
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [6]:
# Preprocess data
# Convert Total Charges into numeric values

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
# invalid value into NaN

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:
# Check missing values
print(df.isnull().sum())

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [8]:
# Fill missing with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [9]:
# Cust Id remove     rem col   make changes in original df
df.drop("customerID", axis=1, inplace=True)

C:\Users\opc\AppData\Local\Temp\2\ipykernel_4088\3401743069.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop("customerID", axis=1, inplace=True)


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [11]:
# Sepearate features and target
X = df.drop("Churn", axis=1)
y = df["Churn"].map({
                      "No" : 0,
                      "Yes" : 1
})
print("Input Features Shape : ", X.shape)
print("Target Shape : ", y.shape)

Input Features Shape :  (7043, 19)
Target Shape :  (7043,)


In [12]:
# Identify numerical and categorical features
numerical_features = X.select_dtypes(
            include=["int64","float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
            include=["object"]
).columns.tolist()

print(numerical_features)
print("---------------------")
print(categorical_features)

C:\Users\opc\AppData\Local\Temp\2\ipykernel_4088\2041482257.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
---------------------
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [13]:
# Split the dataset into training and testing
# train_test_split
# X and y -> X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state = 42, stratify = y)
                     # testset - 20%, trainset - 80%
                     # random_state - same split 
                    # stratify = y, maintain approx same churn/non churn proportion
print("Training Set : ", X_train.shape[0])
print("Testing Set : ", X_test.shape[0])

Training Set :  5634
Testing Set :  1409


In [14]:
# Transformations
# numerical - StandardScaler - higher and lower values should be treated equally
# OneHotEncoder
# ColumnTransformer - apply correct preprocessing to each group

preprocessor = ColumnTransformer(
       transformers = [
           ("num",
             StandardScaler(),
             numerical_features
           ),
           ("cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
           )
       ]
)
print("Preprocessing Pipeline created")

Preprocessing Pipeline created


In [15]:
# Train the LogisticRegression Model
logistic_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter = 1000, class_weight = "balanced")
        )
    ]
)

# Train the Model
logistic_model.fit(X_train, y_train)
print("logistic_model trained")

logistic_model trained


In [16]:
# Evaluate the Model
y_pred = logistic_model.predict(X_test) # predictions done by model
# Actual Values - y_test
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
print("Accuracy : ", accuracy)
print("Precision : ", precision)

Accuracy :  0.7381121362668559
Precision :  0.504302925989673


In [62]:
import pickle
with open("models/best_model.pkl","wb") as file:
    pickle.dump(logistic_model, file)

print("Model saved successfully")

Model saved successfully


In [17]:
rf_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight="balanced")
                                 # no. of decision tress
        )
    ]
)

# Train the Model
rf_model.fit(X_train, y_train)
print("rf model trained")

rf model trained


In [18]:
rf_pred = rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, rf_pred)
precision_rf = precision_score(y_test, rf_pred)
print("Accuracy RF: ", accuracy_rf)
print("Precision RF : ", precision_rf)

Accuracy RF:  0.751596877217885
Precision RF :  0.5214285714285715


In [19]:
# Demonstrate data changes
import pandas as pd
df = pd.read_csv(r"C:\Users\opc\custchurn-mlops\WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Chanfed - Monthly Charges - first few rows - 10 records
df.loc[0:9, "MonthlyCharges"] = (
    df.loc[0:9, "MonthlyCharges"] + 15.0
)

# Save updated dataset
df.to_csv(
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    index=False
)
print("Dataset updated")

Dataset updated


In [20]:
# Demonstrate data changes
import pandas as pd
df = pd.read_csv(r"C:\Users\opc\custchurn-mlops\WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Chanfed - Monthly Charges - first few rows - 10 records
df.loc[0:9, "MonthlyCharges"] = (
    df.loc[0:9, "MonthlyCharges"] + 30.0
)

# Save updated dataset
df.to_csv(
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    index=False
)
print("Dataset updated")

Dataset updated


In [21]:
df_v3 = pd.read_csv(r"C:\Users\opc\churn_v3\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_v2 = pd.read_csv(r"C:\Users\opc\churn_v2\WA_Fn-UseC_-Telco-Customer-Churn.csv")

comparison = pd.DataFrame({
    "Old_MonthlyCharges" : df_v3["MonthlyCharges"],
    "New_MonthlyCharges" : df_v2["MonthlyCharges"]
})

comparison["Difference"] = (
    comparison["New_MonthlyCharges"] - comparison["Old_MonthlyCharges"] )
changed_records = comparison[
    comparison["Difference"] != 0 ]

changed_records.head(20)

,Old_MonthlyCharges,New_MonthlyCharges,Difference
0,74.85,44.85,-30.0
1,101.95,71.95,-30.0
2,98.85,68.85,-30.0
3,87.30,57.30,-30.0
4,115.70,85.70,-30.0
5,144.65,114.65,-30.0
6,134.10,104.10,-30.0
7,74.75,44.75,-30.0
8,149.80,119.80,-30.0
9,101.15,71.15,-30.0


In [22]:
import mlflow
print(mlflow.__version__)

3.18.0


In [23]:
mlflow.set_experiment("Customer Churn Prediction")

<Experiment: artifact_location='file:///C:/Users/opc/custchurn-mlops/mlruns/1', creation_time=1794894079798, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794894079798, lifecycle_stage='active', name='Customer Churn Prediction', tags={}, trace_location=None, workspace='default'>

In [24]:
experiment = mlflow.get_experiment_by_name("Customer Churn Prediction")
print (" Id :", experiment.experiment_id)
print (" Name : " , experiment.name)

 Id : 1
 Name :  Customer Churn Prediction


In [25]:
import mlflow.sklearn
with mlflow.start_run(run_name = "Logistic Regression"):
    # Define Model
    logistic_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter = 1000, class_weight = "balanced")
        )
    ]
    )

    # Train the Model
    logistic_model.fit(X_train, y_train)
    print("logistic_model trained")
    # Evaluate the Model
    y_pred = logistic_model.predict(X_test) # predictions done by model
    # Actual Values - y_test
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    print("Accuracy : ", accuracy)
    print("Precision : ", precision)

    # log parameters
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "balanced")
    # log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    # Log Trained Model
    mlflow.sklearn.log_model(
       logistic_model,
       name="logistic_regression_model"
    )
    print("Logistic Regression Expermient Tracked")

logistic_model trained
Accuracy :  0.7381121362668559
Precision :  0.504302925989673
Logistic Regression Expermient Tracked


In [30]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [36]:
# Run multiple hyperparameter experiments
# Random Forest
n_estimator_values = [50, 100, 200]
max_depth_values = [5, 10, 15]
# total - 9 exps

for n in n_estimator_values:
    for depth in max_depth_values:
        # RF_50_Depth_5
        with mlflow.start_run(run_name = f"RF_{n}_Depth_{depth}"): 
            rf_model = Pipeline(
                        steps = [
                           ("preprocessor", preprocessor),
                               (
            "classifier",
            RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=42, class_weight="balanced")
                                 # no. of decision tress
        )
    ]
)
# Train the Model
            rf_model.fit(X_train, y_train)
            print("rf model trained")

            rf_pred = rf_model.predict(X_test)
            accuracy_rf = accuracy_score(y_test, rf_pred)
            precision_rf = precision_score(y_test, rf_pred)
            print("Accuracy RF: ", accuracy_rf)
            print("Precision RF : ", precision_rf)

# log parameters
            mlflow.log_param("model_type", "Random Forest")
            mlflow.log_param("n_estimators", n),
            mlflow.log_param("max_depth", depth),
            mlflow.log_param("class_weight", "balanced")
    # log metrics
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
    # Log Trained Model
            mlflow.sklearn.log_model(
                logistic_model,
                name="random_forest_model"
    )
print(
                   f"Trees : {n}, Depth : {depth}, "
                   f"Accuracy: {accuracy}",
                   f"precision : {precision}"
)
    

rf model trained
Accuracy RF:  0.7381121362668559
Precision RF :  0.5043177892918825
🏃 View run RF_50_Depth_5 at: http://127.0.0.1:5000/#/experiments/1/runs/04123b0c4f784e05b7cb5ef9cecc115f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
rf model trained
Accuracy RF:  0.7530163236337828
Precision RF :  0.5243445692883895
🏃 View run RF_50_Depth_10 at: http://127.0.0.1:5000/#/experiments/1/runs/a2fd3693b4a64b88af4235eede654a05
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
rf model trained
Accuracy RF:  0.758694109297374
Precision RF :  0.5363247863247863
🏃 View run RF_50_Depth_15 at: http://127.0.0.1:5000/#/experiments/1/runs/df64a7c870e840e783a5fa01e116fa52
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
rf model trained
Accuracy RF:  0.7388218594748048
Precision RF :  0.5051546391752577
🏃 View run RF_100_Depth_5 at: http://127.0.0.1:5000/#/experiments/1/runs/3ae2d0c83a374027bc843f71b132c587
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/

In [34]:
# Register the model :
from mlflow.models import infer_signature

with mlflow.start_run(run_name = "Logistic Regression"):
    # Define Model
    logistic_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter = 1000, class_weight = "balanced")
        )
    ]
    )

    # Train the Model
    logistic_model.fit(X_train, y_train)
    print("logistic_model trained")
    # Evaluate the Model
    y_pred = logistic_model.predict(X_test) # predictions done by model
    # Actual Values - y_test
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    print("Accuracy : ", accuracy)
    print("Precision : ", precision)
    signature = infer_signature(X_train, logistic_model.predict(X_train))
        
    # log parameters
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "balanced")
    # log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    # Log Trained Model
    mlflow.sklearn.log_model(
       logistic_model,
       name="logistic_regression_model",
       signature = signature,
       registered_model_name ="LogisticRegressionModelRegistered."
    )
    print("Logistic Regression Expermient Tracked")

logistic_model trained
Accuracy :  0.7381121362668559
Precision :  0.504302925989673


C:\Users\opc\custchurn-mlops\mlops_venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'LogisticRegressionModelRegistered.' already exists. Creating a new version of this model...
2026/11/18 04:58:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to fini

Logistic Regression Expermient Tracked
🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/1/runs/6669ec03e7ac48e79063f49c3c4a762b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [33]:
# Prepare a sample customer reuqest
# all feature values
# X_test - pick one customer record

import json

sample_customer = json.loads(
    X_test.iloc[[0]].to_json(orient="records") # each row is represented as a seperate JSON
)[0] # extarct the first dictionary from the list 
# one row - treated as a PYthon Dictionary

print(json.dumps(sample_customer, indent=4))

{
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 72,
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Two year",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Credit card (automatic)",
    "MonthlyCharges": 114.05,
    "TotalCharges": 8468.2
}


In [47]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = MlflowClient()

model_name = "LogisticRegressionModelRegistered."

version = client.get_model_version(
    name=model_name,
    version="2"
)

print("Model name:", version.name)
print("Model version:", version.version)
print("Run ID:", version.run_id)
print("Source:", version.source)

Model name: LogisticRegressionModelRegistered.
Model version: 2
Run ID: a3e4c670b22041f2846ddaca30ff334c
Source: models:/m-180dffdeba084a79ba6e6ec3d886e6ba


In [48]:
import mlflow
from mlflow.tracking import MlflowClient

print("MLflow version:", mlflow.__version__)

mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = MlflowClient()

model_id = "m-180dffdeba084a79ba6e6ec3d886e6ba"

# Inspect the logged model
logged_model = client.get_logged_model(model_id)

print("\n--- Logged Model Details ---")
print(logged_model)

# Inspect registered model download URI
download_uri = client.get_model_version_download_uri(
    name="LogisticRegressionModelRegistered.",
    version="2"
)

print("\n--- Model Download URI ---")
print(download_uri)

# Inspect files stored under the model URI
print("\n--- Model Artifacts ---")

try:
    artifacts = mlflow.artifacts.list_artifacts(
        artifact_uri=f"models:/{model_id}"
    )

    for artifact in artifacts:
        print(artifact.path, artifact.is_dir)

except Exception as e:
    print("Artifact listing error:", repr(e))

MLflow version: 3.18.0

--- Logged Model Details ---
LoggedModel(artifact_location='file:///C:/Users/opc/custchurn-mlops/mlruns/1/models/m-180dffdeba084a79ba6e6ec3d886e6ba/artifacts', creation_timestamp=1794977776232, experiment_id='1', last_updated_timestamp=1794977787505, model_id='m-180dffdeba084a79ba6e6ec3d886e6ba', model_type='', model_uri='models:/m-180dffdeba084a79ba6e6ec3d886e6ba', name='logistic_regression_model', source_run_id='a3e4c670b22041f2846ddaca30ff334c', status=<LoggedModelStatus.READY: 'READY'>, status_message='')

--- Model Download URI ---
file:///C:/Users/opc/custchurn-mlops/mlruns/1/models/m-180dffdeba084a79ba6e6ec3d886e6ba/artifacts

--- Model Artifacts ---
MLmodel False
conda.yaml False
model.skops False
python_env.yaml False
requirements.txt False


In [63]:
import mlflow
import mlflow.sklearn

from mlflow.tracking import MlflowClient

# Connect to MLflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = MlflowClient()

# Load your existing registered model
old_model_uri = "models:/LogisticRegressionModelRegistered./4"

model = mlflow.sklearn.load_model(old_model_uri)

print("Existing model loaded successfully!")

experiment_name= "Customer Churn Prediction - Rancher Proxy"

experiment = mlflow.set_experiment(experiment_name)

print("Experiment ID:", experiment.experiment_id)

print("Artifact location:",
      client.get_experiment(experiment.experiment_id).artifact_location)

Existing model loaded successfully!
Experiment ID: 2
Artifact location: file:///C:/Users/opc/custchurn-mlops/mlartifacts/2


In [65]:
with mlflow.start_run(
    run_name="Logistic Regression - Rancher Ready"
):

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name="LogisticRegressionModelRegistered."
    )

    print("New logged model ID:", model_info.model_id)

Registered model 'LogisticRegressionModelRegistered.' already exists. Creating a new version of this model...
2026/11/18 09:40:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LogisticRegressionModelRegistered., version 6
Created version '6' of model 'LogisticRegressionModelRegistered.'.


New logged model ID: m-0335feebaf9e4de78cdf640b1fe776f2
🏃 View run Logistic Regression - Rancher Ready at: http://127.0.0.1:5000/#/experiments/2/runs/d6098e484d2d45379b61919b09c89f34
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [58]:
%%writefile .github/workflows/mlops-pipeline.yml

name: MLOps CI Pipeline

on:
  push:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run tests
        run: pytest

Writing .github/workflows/mlops-pipeline.yml


In [57]:
import os
print (os.getcwd())

C:\Users\opc\custchurn-mlops


In [60]:
%%writefile tests/test_model.py

import joblib
import pandas as pd

def test_model_prediction():

    model = joblib.load(
        "models/best_model.pkl"
    )

    sample_data = pd.DataFrame([{
        "gender": "Female",
        "SeniorCitizen": 0,
        "Partner": "Yes",
        "Dependents": "No",
        "tenure": 12,
        "PhoneService": "Yes",
        "MultipleLines": "No",
        "InternetService": "DSL",
        "OnlineSecurity": "No",
        "OnlineBackup": "Yes",
        "DeviceProtection": "No",
        "TechSupport": "No",
        "StreamingTV": "No",
        "StreamingMovies": "No",
        "Contract": "Month-to-month",
        "PaperlessBilling": "Yes",
        "PaymentMethod": "Electronic check",
        "MonthlyCharges": 50.0,
        "TotalCharges": 600.0
    }])

    prediction = model.predict(sample_data)

    assert len(prediction) == 1

    assert prediction[0] in [0, 1]

Writing tests/test_model.py


In [66]:
%%writefile tests/test_data.py

import pandas as pd

def test_dataset_columns():

    df = pd.read_csv(
        "WA_Fn-UseC_-Telco-Customer-Churn.csv"
    )

    required_columns = [
        "customerID",
        "gender",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "Churn"
    ]

    for column in required_columns:

        assert column in df.columns

Writing tests/test_data.py
